# 02 — Transform: Bronze → Silver

Joins customers + orders, adds computed columns, writes to **silver**.

In [ ]:
import sys
sys.path.insert(0, '/workspace')
from scripts.spark_session import get_spark, BRONZE, SILVER
from pyspark.sql import functions as F
spark = get_spark('02_transform')

In [ ]:
customers = spark.read.parquet(f'{BRONZE}/customers')
orders = spark.read.parquet(f'{BRONZE}/orders')

enriched = (
    orders
    .join(customers, on='customer_id', how='inner')
    .withColumn('line_total', F.col('quantity') * F.col('unit_price'))
    .withColumn('order_month', F.date_format('order_date', 'yyyy-MM'))
)
enriched.show(truncate=False)

In [ ]:
enriched.write.mode('overwrite').partitionBy('order_month').parquet(f'{SILVER}/order_details')
print(f'✅ Wrote {enriched.count()} enriched rows to silver')